In [26]:
pip install sktime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 130.9 MB/s eta 0:00:0000:01
Note: you may need to restart the kernel to use updated packages.


In [35]:
import pandas as pd
import numpy as np
from decimal import Decimal, ROUND_HALF_UP
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sktime.classification.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings('ignore')

In [3]:
case_df = pd.read_csv('s3://ads508-s3/time_series_covid19_confirmed_US.csv')
case_df.head()

,UID,iso2,iso3,code3,FIPS,Admin2,Province_State,Country_Region,Lat,Long_,...,2/28/23,3/1/23,3/2/23,3/3/23,3/4/23,3/5/23,3/6/23,3/7/23,3/8/23,3/9/23
0,84001001,US,USA,840,1001.0,Autauga,Alabama,US,32.539527,-86.644082,...,19732,19759,19759,19759,19759,19759,19759,19759,19790,19790
1,84001003,US,USA,840,1003.0,Baldwin,Alabama,US,30.727750,-87.722071,...,69641,69767,69767,69767,69767,69767,69767,69767,69860,69860
2,84001005,US,USA,840,1005.0,Barbour,Alabama,US,31.868263,-85.387129,...,7451,7474,7474,7474,7474,7474,7474,7474,7485,7485
3,84001007,US,USA,840,1007.0,Bibb,Alabama,US,32.996421,-87.125115,...,8067,8087,8087,8087,8087,8087,8087,8087,8091,8091
4,84001009,US,USA,840,1009.0,Blount,Alabama,US,33.982109,-86.567906,...,18616,18673,18673,18673,18673,18673,18673,18673,18704,18704


In [4]:
death_df = pd.read_csv('s3://ads508-s3/time_series_covid19_deaths_US.csv')
death_df.head()

,UID,iso2,iso3,code3,FIPS,Admin2,Province_State,Country_Region,Lat,Long_,...,2/28/23,3/1/23,3/2/23,3/3/23,3/4/23,3/5/23,3/6/23,3/7/23,3/8/23,3/9/23
0,84001001,US,USA,840,1001.0,Autauga,Alabama,US,32.539527,-86.644082,...,230,232,232,232,232,232,232,232,232,232
1,84001003,US,USA,840,1003.0,Baldwin,Alabama,US,30.727750,-87.722071,...,724,726,726,726,726,726,726,726,727,727
2,84001005,US,USA,840,1005.0,Barbour,Alabama,US,31.868263,-85.387129,...,103,103,103,103,103,103,103,103,103,103
3,84001007,US,USA,840,1007.0,Bibb,Alabama,US,32.996421,-87.125115,...,109,109,109,109,109,109,109,109,109,109
4,84001009,US,USA,840,1009.0,Blount,Alabama,US,33.982109,-86.567906,...,261,261,261,261,261,261,261,261,261,261


In [5]:
case_df.columns

Index(['UID', 'iso2', 'iso3', 'code3', 'FIPS', 'Admin2', 'Province_State',
       'Country_Region', 'Lat', 'Long_',
       ...
       '2/28/23', '3/1/23', '3/2/23', '3/3/23', '3/4/23', '3/5/23', '3/6/23',
       '3/7/23', '3/8/23', '3/9/23'],
      dtype='object', length=1154)

In [6]:
print(case_df.shape)
print(death_df.shape)

(3342, 1154)
(3342, 1155)


In [7]:
case_df = case_df.drop(
    columns=['UID', 'iso2', 'iso3', 'code3', 'FIPS', 'Country_Region', 'Combined_Key'],
    errors='ignore'
)

# Drop rows where 'Admin2' is missing
case_df = case_df.dropna(subset=['Admin2'])

# Reshape the dataframe from wide to long
case_df = pd.melt(
    case_df,
    id_vars=['Admin2', 'Province_State', 'Lat', 'Long_'],
    var_name='Date',
    value_name='Deaths'  # Use different name to avoid conflict
)

# Convert 'Date' column with explicit format
case_df['Date'] = pd.to_datetime(case_df['Date'], format='%m/%d/%y')

# Optimize memory usage
case_df['Deaths'] = pd.to_numeric(case_df['Deaths'], downcast='integer')
case_df['Lat'] = np.round(case_df['Lat'], 3)
case_df['Long_'] = np.round(case_df['Long_'], 3)
case_df['Admin2'] = case_df['Admin2'].astype('category')
case_df['Province_State'] = case_df['Province_State'].astype('category')

# Resample weekly death data
case_df.set_index('Date', inplace=True)
case_resample = case_df.groupby(
    ['Admin2', 'Lat', 'Long_', 'Province_State'],
    observed=False
).resample('W')['Deaths'].sum().reset_index()

# Final cleanup
case_resample.reset_index(drop=True, inplace=True)

In [8]:
death_df = death_df.drop(
    columns=['UID', 'iso2', 'iso3', 'code3', 'FIPS', 'Country_Region', 'Combined_Key'],
    errors='ignore'
)

# Drop rows where 'Admin2' is missing
death_df = death_df.dropna(subset=['Admin2'])

# Reshape the DataFrame from wide to long
death_df = pd.melt(
    death_df,
    id_vars=['Admin2', 'Province_State', 'Lat', 'Long_', 'Population'],
    var_name='Date',
    value_name='Confirmed'
)

# Convert 'Date' column with explicit format
death_df['Date'] = pd.to_datetime(death_df['Date'], format='%m/%d/%y')

# Optimize memory usage
death_df['Confirmed'] = pd.to_numeric(death_df['Confirmed'], downcast='integer')
death_df['Population'] = pd.to_numeric(death_df['Population'], downcast='integer')
death_df['Lat'] = np.round(death_df['Lat'], 3)
death_df['Long_'] = np.round(death_df['Long_'], 3)
death_df['Admin2'] = death_df['Admin2'].astype('category')
death_df['Province_State'] = death_df['Province_State'].astype('category')

# Resample weekly confirmed data
death_df.set_index('Date', inplace=True)
death_resample = death_df.groupby(
    ['Admin2', 'Lat', 'Long_', 'Province_State', 'Population'],
    observed=False
).resample('W')['Confirmed'].sum().reset_index()

# Final cleanup
death_resample.reset_index(drop=True, inplace=True)

In [9]:
death_staging = death_resample[['Date', 'Admin2', 'Lat', 'Long_', 'Province_State', 'Population', 'Confirmed']]

# Merge on full key set to avoid many-to-many explosion
base_df = pd.merge(
    case_resample,
    death_staging,
    on=['Date', 'Admin2', 'Lat', 'Long_', 'Province_State'],
    how='left'
)

base_df.head()

,Admin2,Lat,Long_,Province_State,Date,Deaths,Population,Confirmed
0,Abbeville,34.223,-82.462,South Carolina,2020-01-26,0,24527,0
1,Abbeville,34.223,-82.462,South Carolina,2020-02-02,0,24527,0
2,Abbeville,34.223,-82.462,South Carolina,2020-02-09,0,24527,0
3,Abbeville,34.223,-82.462,South Carolina,2020-02-16,0,24527,0
4,Abbeville,34.223,-82.462,South Carolina,2020-02-23,0,24527,0


In [31]:
base_df.shape

(547104, 8)

In [14]:
print(base_df['Date'].min())
print(base_df['Date'].max())

2020-01-26 00:00:00
2023-03-12 00:00:00


In [32]:
"""The following are the top 10 states with the most covid deaths between 01-26-2020 and 03-12-2023... The numbers are not adding up properly need to look more into this"""
(
    base_df
    .groupby('Province_State', observed=False)['Confirmed']
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

Province_State
California      65490302
Texas           61302166
New York        58121236
Florida         51475342
Pennsylvania    31912144
Illinois        28240376
New Jersey      28101090
Georgia         26228841
Ohio            26072614
Michigan        25546398
Name: Confirmed, dtype: int32

In [34]:
y = base_df['Confirmed']
X = base_df

In [36]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.20)

In [37]:
models = {
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    'LightGBM': LGBMClassifier(),
    'DecisionTree': DecisionTreeClassifier(),
    'Bagging': BaggingClassifier()
}

def train_model(model, X_train, y_train):
    model.fit(X_train, y_train)
    return model
    
trained_models = {}
for name, model in models.items():
    print(f"Training {name}...")
    trained_models[name] = train_model(model, X_train, y_train)

TypeError: BaggingClassifier.__init__() missing 1 required positional argument: 'estimator'